# MODUL PRAKTIKUM BIG DATA
## Pertemuan 9 — Basis Data NoSQL: Instalasi MongoDB & Integrasi dengan PySpark (Arch Linux Safe Edition)

| Informasi | Keterangan |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | S1 Teknologi Informasi — Universitas Tidar |
| **Dosen Pengampu** | Alifia Revan Prananda, S.Pd., M.Eng. |
| **Sistem Operasi** | Arch Linux / CachyOS (Userspace Zero-Conflict Environment) |
| **Praktikan** | Sholahuddin Ahmad (NPM: 250506070) |
| **Software Stack** | MongoDB Community 8.x, mongosh, Apache Spark 3.5.9, OpenJDK 11, uv (Python 3.11) |
| **Topik** | Konsep NoSQL, Instalasi MongoDB (AUR), Operasi CRUD, Integrasi MongoDB-PySpark |
| **Estimasi Waktu** | 150 menit |
| **Prasyarat** | Modul Pertemuan 1-7 selesai, **UTS (Pertemuan 8) telah dilaksanakan** |

---

### 🛡️ Jaminan Keamanan Sistem & Cara Menggunakan Modul Ini
Modul ini telah disesuaikan khusus untuk lingkungan **Arch Linux / CachyOS**:
1. **Instalasi Native AUR**: MongoDB Community di Arch Linux diinstal menggunakan paket AUR pre-compiled (`mongodb-bin` dan `mongosh-bin`) via `paru` — menghindari kompilasi source code yang memakan waktu berjam-jam.
2. **Perbedaan Service Unit**: Di Arch Linux, unit systemd bernama **`mongodb.service`** (bukan `mongod.service` seperti di Ubuntu).
3. **Environment Python Modern (`uv`)**: Menggunakan virtual environment proyek (`.venv`) yang dikelola oleh `uv`, bukan Conda global.
4. **Isolasi Java Runtime**: Integrasi PySpark tetap menggunakan OpenJDK 11 terisolasi lewat `spark_init.py` agar tidak bentrok dengan Java bawaan sistem.

---
## Recap & Selamat Melewati UTS!

Selamat, anda baru saja menyelesaikan UTS yang mencakup Pertemuan 1-7. Mulai pertemuan ini kita membuka topik baru dalam ekosistem Big Data: **penyimpanan data NoSQL**.

- [ ] Arch Linux aktif, virtual environment proyek aktif via `uv` / `.venv`
- [ ] Hadoop aktif jika dibutuhkan (`start-dfs.sh`, `start-yarn.sh`) — pertemuan ini **tidak wajib** menyalakan Hadoop di awal, namun klaster tetap siap digunakan.

## Tujuan Pembelajaran

Setelah menyelesaikan Pertemuan 9, mahasiswa mampu:
1. Menjelaskan konsep basis data NoSQL dan perbedaannya dengan basis data relasional.
2. Menginstall MongoDB Community Server dan MongoDB Shell (`mongosh`) di Arch Linux.
3. Melakukan operasi CRUD (*Create, Read, Update, Delete*) menggunakan `mongosh` dan PyMongo.
4. Mengintegrasikan data dari MongoDB ke dalam Spark DataFrame untuk dianalisis lebih lanjut secara aman dan terisolasi.

---

## 9.1 Mengapa NoSQL? Mengingat Kembali Konsep *Variety* dari Big Data

Ingat kembali kerangka **5V** pada Pertemuan 3 — salah satunya adalah **Variety** (keragaman format data). Sejauh ini, seluruh data yang kita olah (CSV, Parquet) bersifat **terstruktur**, mengikuti skema tabel yang kaku (baris & kolom tetap). Di dunia nyata, banyak data justru **semi-terstruktur** dan skemanya dapat bervariasi antar-baris — misalnya data profil pengguna aplikasi, di mana sebagian pengguna memiliki data alamat lengkap sementara yang lain tidak, atau setiap produk pada katalog e-commerce memiliki atribut yang sama sekali berbeda (baju punya atribut "ukuran", elektronik punya atribut "voltase").

| Aspek | Basis Data Relasional (SQL) | Basis Data NoSQL (mis. MongoDB) |
|---|---|---|
| **Struktur data** | Tabel dengan skema tetap (kolom harus didefinisikan di awal) | Dokumen (mirip JSON) dengan skema fleksibel |
| **Relasi antar data** | Menggunakan JOIN antar tabel | Data terkait dapat disematkan (*embedded*) langsung dalam satu dokumen |
| **Skalabilitas** | Umumnya vertikal (tambah kapasitas 1 server) | Dirancang untuk skalabilitas horizontal (tambah banyak server) |
| **Kapan digunakan** | Data terstruktur, butuh konsistensi transaksi ketat | Data semi-terstruktur, skema sering berubah, butuh skala sangat besar |

>**Bukan pengganti, tapi pelengkap.** Dalam ekosistem Big Data nyata, HDFS/Spark (data terstruktur skala besar) dan MongoDB (data semi-terstruktur, akses cepat per-dokumen) sering dipakai **bersamaan**, masing-masing untuk kebutuhan yang paling sesuai — bukan saling menggantikan.

**MongoDB** adalah basis data NoSQL bertipe *document store* paling populer, menyimpan data dalam format **BSON** (Binary JSON) yang terorganisir dalam **collection** (setara "tabel") berisi **document** (setara "baris", namun berbentuk seperti objek JSON).

---
## 9.2 Instalasi MongoDB Community Server di Arch Linux

> [!NOTE]
> **Mengapa Berbeda dengan Ubuntu?**
> Pada modul aslinya (Ubuntu), MongoDB dipasang melalui repository apt `repo.mongodb.org`. Namun di Arch Linux, MongoDB telah dihapus dari repositori resmi (`pacman`) sejak 2019 akibat perubahan lisensi ke SSPL (*Server Side Public License*).
> Di Arch Linux, instalasi resmi dilakukan melalui **AUR (Arch User Repository)** menggunakan helper `paru`. Gunakan varian `-bin` agar menggunakan biner resmi yang sudah dikompilasi, sehingga tidak perlu mengkompilasi source code C++ MongoDB yang memakan waktu 4+ jam.

**Jalankan langkah berikut di Terminal Arch Linux (Alacritty / Kitty / Fish Shell):**

**Langkah 1: Pasang MongoDB Server & mongosh via AUR**

```bash
paru -S mongodb-bin mongosh-bin
```

**Langkah 2: Jalankan MongoDB Service**

> [!IMPORTANT]
> **Perhatikan Nama Unit Service!**
> Di Ubuntu servicenya bernama `mongod.service`, sedangkan di Arch Linux paket AUR menggunakan nama unit **`mongodb.service`** (bukan `mongod`).

```bash
# Nyalakan MongoDB dan atur autostart saat booting
sudo systemctl start mongodb
sudo systemctl enable mongodb
```

**Langkah 3: Verifikasi Service Berjalan**

```bash
sudo systemctl status mongodb
```

Cari tulisan `Active: active (running)` berwarna hijau. Tekan `q` untuk keluar dari status viewer.

Verifikasi biner yang terpasang:
```bash
mongod --version
```

---

## 9.3 Operasi Dasar dengan `mongosh` (MongoDB Shell)

`mongosh` adalah *command-line interface* resmi untuk berinteraksi langsung dengan MongoDB — mirip peran `hdfs dfs` untuk HDFS atau `psql` untuk PostgreSQL.

**Masuk ke MongoDB shell:**

```bash
mongosh
```

Anda akan masuk ke prompt interaktif `test>`. Coba jalankan perintah-perintah dasar berikut **di dalam `mongosh`**:

```javascript
// Berpindah/membuat database bernama "praktikum_bigdata"
use praktikum_bigdata

// Menyisipkan satu dokumen ke collection "produk" (otomatis dibuat jika belum ada)
db.produk.insertOne({ nama_produk: "Kopi Susu", kategori: "Minuman", harga: 18000, stok: 120 })

// Menampilkan seluruh dokumen pada collection "produk"
db.produk.find()

// Menampilkan dengan format lebih rapi
db.produk.find().pretty()
```

Keluar dari `mongosh` dengan mengetik:
```javascript
exit
```

>**Perhatikan:** kita tidak perlu mendefinisikan struktur/skema `produk` terlebih dahulu seperti pada tabel SQL — MongoDB otomatis membuat collection saat dokumen pertama disisipkan. Inilah wujud nyata dari *flexible schema* yang dibahas pada Sub-bab 9.1.

---
## Melanjutkan di Jupyter Notebook (Arch Linux Safe Environment)

Di sistem Arch Linux kita yang menggunakan manajemen virtual environment modern berbasis `uv`, driver PyMongo cukup dipasang ke `.venv` proyek tanpa perlu Conda:

```bash
cd /mnt/alterra/repos/praktikum-big-data
uv pip install pymongo
```

Verifikasi instalasi driver PyMongo:
```bash
.venv/bin/python -c "import pymongo; print('PyMongo terpasang versi:', pymongo.__version__)"
```

Lalu jalankan Jupyter Notebook dari workspace proyek:
```bash
uv run jupyter notebook
```

---
## 9.4 Koneksi ke MongoDB dari Python

In [ ]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB lokal (port default 27017)
client = MongoClient("mongodb://localhost:27017/")

# Memilih (atau otomatis membuat) database
db = client["praktikum_bigdata"]

# Memilih (atau otomatis membuat) collection
koleksi_produk = db["produk"]

print("⚡ Koneksi ke MongoDB lokal berhasil!")
print("Daftar database aktif:", client.list_database_names())

Jika muncul daftar nama database tanpa error, koneksi PyMongo ke MongoDB anda telah berhasil.

---

## 9.5 Operasi CRUD dengan PyMongo

### 9.5.1 Create — Menyisipkan Dokumen

In [ ]:
# Menghapus data lama jika ada (agar percobaan kita mulai dari kondisi bersih)
koleksi_produk.delete_many({})

# insert_one() — menyisipkan satu dokumen
koleksi_produk.insert_one({
    "nama_produk": "Kopi Susu",
    "kategori": "Minuman",
    "harga": 18000,
    "stok": 120,
    "tags": ["best-seller", "dingin"]
})

# insert_many() — menyisipkan banyak dokumen sekaligus
koleksi_produk.insert_many([
    {"nama_produk": "Roti Bakar", "kategori": "Makanan", "harga": 15000, "stok": 45, "tags": ["hangat"]},
    {"nama_produk": "Es Teh", "kategori": "Minuman", "harga": 8000, "stok": 200, "tags": ["dingin", "murah"]},
    {"nama_produk": "Kentang Goreng", "kategori": "Makanan", "harga": 20000, "stok": 60, "tags": ["gurih"]},
    {"nama_produk": "Cappuccino", "kategori": "Minuman", "harga": 25000, "stok": 80, "tags": ["hangat", "best-seller"]},
])

print("Jumlah dokumen di collection 'produk':", koleksi_produk.count_documents({}))

> **Perhatikan** setiap produk memiliki jumlah `tags` yang berbeda-beda — sebagian punya 1 tag, sebagian punya 2. Ini **tidak mungkin dilakukan secara alami** pada tabel SQL biasa tanpa tabel relasi tambahan, namun sangat wajar pada model dokumen MongoDB.

### 9.5.2 Read — Membaca Dokumen

In [ ]:
# find_one() — mengambil satu dokumen pertama yang cocok
satu_produk = koleksi_produk.find_one({"kategori": "Minuman"})
print("Satu produk kategori Minuman:", satu_produk)

print()

# find() — mengambil SEMUA dokumen yang cocok (mengembalikan cursor)
print("Seluruh produk kategori Minuman:")
for p in koleksi_produk.find({"kategori": "Minuman"}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

print()

# Filter dengan operator perbandingan: harga lebih dari 15000
print("Produk dengan harga > 15000:")
for p in koleksi_produk.find({"harga": {"$gt": 15000}}):
    print(f"  - {p['nama_produk']} (Rp{p['harga']:,})")

### 9.5.3 Update — Memperbarui Dokumen

In [ ]:
# update_one() — memperbarui SATU dokumen pertama yang cocok
koleksi_produk.update_one(
    {"nama_produk": "Es Teh"},
    {"$set": {"harga": 9000}}   # menaikkan harga Es Teh
)

# update_many() — memperbarui SEMUA dokumen yang cocok
# Contoh: mengurangi stok seluruh produk kategori Minuman sebanyak 5 (simulasi penjualan)
koleksi_produk.update_many(
    {"kategori": "Minuman"},
    {"$inc": {"stok": -5}}   # $inc mengubah nilai relatif (increment/decrement)
)

print("Setelah update:")
for p in koleksi_produk.find({}, {"_id": 0, "nama_produk": 1, "harga": 1, "stok": 1}):
    print(f"  - {p['nama_produk']}: harga=Rp{p['harga']:,}, stok={p['stok']}")

### 9.5.4 Delete — Menghapus Dokumen

In [ ]:
# delete_one() — menghapus SATU dokumen pertama yang cocok
koleksi_produk.delete_one({"nama_produk": "Kentang Goreng"})

print("Jumlah dokumen tersisa:", koleksi_produk.count_documents({}))
print("Daftar produk tersisa:", [p["nama_produk"] for p in koleksi_produk.find({}, {"nama_produk": 1, "_id": 0})])

---
## 9.6 Mengintegrasikan Data MongoDB ke Spark DataFrame (Zero-Conflict Arch Linux)

Setelah data tersimpan di MongoDB, kita sering perlu menganalisisnya menggunakan kemampuan Spark yang sudah dipelajari sejak Pertemuan 4-7 (agregasi skala besar, join dengan sumber data lain, dsb.).

Cara paling sederhana dan tanpa perlu instalasi jar tambahan: **ambil data dari MongoDB dengan PyMongo, ubah menjadi pandas DataFrame, lalu konversi menjadi Spark DataFrame**.

> [!IMPORTANT]
> **Isolasi Java & Konversi BSON di Arch Linux:**
> 1. Gunakan helper `spark_init` yang mengisolasi OpenJDK 11 (`/usr/lib/jvm/java-11-openjdk`) agar SparkSession tidak bentrok dengan Java versi host Arch Linux.
> 2. Kolom `_id` tipe BSON `ObjectId` wajib di-cast menjadi `string` karena PySpark tidak memiliki parser BSON bawaan.
> 3. Kolom array seperti `tags` diratakan atau diubah menjadi string agar terhindar dari ketidakcocokan tipe inferensi schema.

In [ ]:
# Mengimpor modul spark_init yang telah terisolasi aman di Arch Linux
from spark_init import spark, F
import pandas as pd

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dict
dokumen_mongo = list(koleksi_produk.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: Penyesuaian tipe data khusus MongoDB
# _id (ObjectId BSON) diubah menjadi string biasa
df_pandas["_id"] = df_pandas["_id"].astype(str)

# Kolom tags (list) digabungkan menjadi string dipisah koma
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4: Konversi ke Spark DataFrame
df_spark_produk = spark.createDataFrame(df_pandas)

print("=== SKEMA SPARK DATAFRAME ===")
df_spark_produk.printSchema()

print("=== DATA PRODUK DARI MONGODB DI SPARK ===")
df_spark_produk.show(truncate=False)

Sekarang data yang tadinya tersimpan di MongoDB sudah menjadi Spark DataFrame biasa — dapat langsung diolah menggunakan seluruh fungsi PySpark (`groupBy`, `join`, window function, disimpan sebagai Parquet ke HDFS, dan seterusnya).

In [ ]:
# Contoh analisis sederhana: rata-rata harga & jumlah produk per kategori
df_spark_produk.groupBy("kategori").agg(
    F.avg("harga").alias("rata_rata_harga"),
    F.count("nama_produk").alias("jumlah_produk")
).show()

print("Analisis Spark atas data MongoDB selesai.")

> **Untuk skala produksi:** pendekatan "PyMongo → pandas → Spark" di atas cocok untuk data berukuran sedang (ribuan hingga jutaan dokumen). Untuk integrasi skala sangat besar (miliaran dokumen), tim data engineering biasanya menggunakan **MongoDB Spark Connector** resmi (`org.mongodb.spark:mongo-spark-connector`) yang memungkinkan Spark membaca partisi MongoDB secara paralel terdistribusi langsung dari HDFS/klaster tanpa melewati driver lokal.

---
## Latihan Mandiri (Arch Linux Solution)

Pastikan `client`, `db`, dan `koleksi_produk` dari Sub-bab 9.4 masih aktif sebelum menjalankan latihan berikut.

**Soal 1.** Sisipkan (`insert_many`) lima dokumen baru ke collection `produk` dengan kategori bebas (boleh kategori baru selain Makanan/Minuman, misalnya "Snack" dan "Pastry").

In [ ]:
# Jawaban Soal 1
produk_tambahan = [
    {"nama_produk": "Keripik Singkong", "kategori": "Snack", "harga": 12000, "stok": 50, "tags": ["renyah", "gurih"]},
    {"nama_produk": "Croissant Cokelat", "kategori": "Pastry", "harga": 22000, "stok": 30, "tags": ["manis", "hangat"]},
    {"nama_produk": "Kacang Almond Panggang", "kategori": "Snack", "harga": 28000, "stok": 25, "tags": ["sehat"]},
    {"nama_produk": "Danish Keju", "kategori": "Pastry", "harga": 24000, "stok": 20, "tags": ["gurih", "lembut"]},
    {"nama_produk": "Makaroni Pedas", "kategori": "Snack", "harga": 10000, "stok": 80, "tags": ["pedas", "murah"]}
]

hasil_insert = koleksi_produk.insert_many(produk_tambahan)
print(f"Berhasil menambahkan {len(hasil_insert.inserted_ids)} dokumen baru.")
print("Total dokumen sekarang:", koleksi_produk.count_documents({}))

**Soal 2.** Gunakan `find()` dengan operator `$lt` (kurang dari) untuk menampilkan seluruh produk dengan `harga` di bawah 20000.

In [ ]:
# Jawaban Soal 2
print("Daftar produk dengan harga < Rp20.000:")
cursor_murah = koleksi_produk.find({"harga": {"$lt": 20000}})

for p in cursor_murah:
    tags_str = ", ".join(p.get("tags", []))
    print(f"- {p['nama_produk']} | Kategori: {p['kategori']} | Harga: Rp{p['harga']:,} | Tags: [{tags_str}]")

**Soal 3.** Gunakan `update_many()` untuk menambahkan field baru bernama `diskon` bernilai `0` ke **seluruh** dokumen pada collection `produk` (petunjuk: gunakan operator `$set`).

In [ ]:
# Jawaban Soal 3
hasil_update = koleksi_produk.update_many(
    {},                         # filter kosong = berlaku untuk semua dokumen
    {"$set": {"diskon": 0}}     # menetapkan default diskon = 0
)

print(f"Jumlah dokumen yang diperbarui: {hasil_update.modified_count}")

# Tampilkan sampel verifikasi 3 dokumen
print("\nSampel 3 dokumen setelah penambahan field 'diskon':")
for p in koleksi_produk.find().limit(3):
    print(f"- {p['nama_produk']} -> Diskon: {p.get('diskon')}%")

**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: berikan **satu contoh skenario nyata** (boleh di luar konteks e-commerce yang sering kita pakai) di mana MongoDB akan menjadi pilihan lebih tepat dibanding tabel relasional biasa, dan jelaskan alasannya berdasarkan konsep *flexible schema*. Tulis jawaban pada markdown cell di bawah ini.

### Jawaban Refleksi Soal 4:
Salah satu contoh nyata yang sangat cocok menggunakan MongoDB adalah **Sistem Rekam Medis Elektronik Pasien (EHR / Electronic Health Records) di Rumah Sakit**. 

Setiap spesialisasi medis memiliki parameter pemeriksaan yang sangat bervariasi; misalnya pasien kardiologi memiliki rekaman data elektrokardiogram (EKG) dan tekanan darah harian, pasien oftalmologi memiliki visus mata dan ketebalan kornea, sedangkan pasien imunologi memiliki daftar puluhan alergen spesifik. 

Dengan konsep *flexible schema* (*document-based*) pada MongoDB, seluruh riwayat pemeriksaan dapat disimpan dalam dokumen JSON tunggal tanpa memaksa adanya ratusan kolom bernilai `NULL` atau skema tabel relasional kompleks (*Entity-Attribute-Value*) yang memperlambat query JOIN.

---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 10 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Platform "Kopi Nusantara" (skenario yang sama dari Tugas Mandiri Pertemuan 7) ingin mulai menyimpan **katalog menu** mereka di MongoDB — karena setiap cabang punya menu musiman/spesial dengan atribut yang berbeda-beda (tidak semua menu punya atribut yang sama, misalnya hanya menu kopi yang punya atribut "tingkat_kepahitan", sementara menu makanan punya atribut "pedas_level"). Kalian ditugaskan membangun dan menganalisis katalog ini.

### Instruksi Pengerjaan

Gunakan template notebook **`tugas.ipynb`** (atau `Tugas9_250506070_Sholahuddin_Ahmad.ipynb`), koneksikan ke MongoDB, lalu kerjakan bagian **A sampai E** berikut:

---

**A. Membangun Katalog (Create)** *(bobot 20%)*
Buat database baru bernama `kopi_nusantara`, collection bernama `menu`. Sisipkan **minimal 12 dokumen** menu dengan field bervariasi sesuai kategorinya:
- Menu kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Menu kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Boleh menambahkan field lain sesuai kreativitas anda (mis. `stok`, `rekomendasi_barista`).

**B. Kueri Bertingkat (Read)** *(bobot 20%)*
Tampilkan hasil dari **tiga** kueri berikut:
1. Seluruh menu kategori "Kopi" dengan `tingkat_kepahitan` di atas 3.
2. Seluruh menu dengan `harga` di antara 15000 dan 30000.
3. Menu yang **tidak memiliki** field `pedas_level` sama sekali (gunakan query `{"pedas_level": {"$exists": False}}`).

**C. Pembaruan Data (Update)** *(bobot 15%)*
Tambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`. Kemudian, ubah `status` menjadi `"Habis"` khusus untuk 2 menu pilihan anda menggunakan `update_one()`.

**D. Integrasi ke Spark** *(bobot 25%)*
Ambil seluruh data dari collection `menu`, ubah menjadi Spark DataFrame:
1. `printSchema()` hasil konversi (perhatikan kolom `tingkat_kepahitan` dan `pedas_level` yang otomatis bernilai null untuk kategori yang berlawanan).
2. Rata-rata `harga` per `kategori` menggunakan `groupBy`.

**E. Kesimpulan** *(bobot 20%)*
Tulis pada markdown cell (**minimal 100 kata**): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?